# Advanced Python Object Finalization: 21 Problems with Worked Solutions

**Topics:** `__del__`, name deletion versus object lifetime, CPython reference counting, container/closure references, traceback retention, cyclic garbage collection, finalizer resurrection, unraisable exceptions, `weakref`, deterministic cleanup, context managers, `ExitStack`, and thread-safe close.

**Based on the supplied lesson on `__del__`.** The original lesson demonstrates that `del` removes a binding, that retained tracebacks can keep objects alive, that finalizer exceptions are ignored by normal callers and reported via `sys.stderr`, and that context managers are preferable for critical cleanup. This notebook develops those ideas into harder, independently runnable exercises and adds explicitly marked modern-Python extensions.

### How to work through this notebook
1. Read a **Challenge** and write your own attempt in a scratch cell *before* inspecting the solution.
2. Read the worked reasoning, run the solution cell, then run its assertion cell.
3. For maximum reproducibility, use **Restart Kernel and Run All**. Cells use small functions to keep temporary references out of notebook output history.
4. Study the further challenge, then adapt the example and add another assertion.

**Runtime:** Python 3.10+ and standard library only. Some checks explicitly require **CPython**, whose reference counting often triggers finalizers immediately. Other Python implementations and notebook frontends may retain objects longer. `gc.collect()` requests a collection; it is **not** a general contract guaranteeing when an arbitrary finalizer runs. Never use a finalizer to guarantee a file, socket, or transaction is closed.

**Safety and accuracy:** We deliberately do **not** reproduce the source's `ctypes.c_long.from_address(id(obj))` helper: dereferencing an invalid/stale address can crash the interpreter, and CPython object internals and refcounts are implementation details. We use `weakref.ref`, carefully scoped examples, and `sys.getrefcount` for the one CPython-specific counting experiment. We do not rely on a specific order of finalizer calls, `id` reuse, or interpreter-shutdown behavior.

In [1]:
import contextlib
import gc
import io
import platform
import sys
import threading
import traceback
import weakref

IS_CPYTHON = platform.python_implementation() == "CPython"
print(f"Python {sys.version.split()[0]} | implementation: {platform.python_implementation()}")
print("All exercises use the standard library; CPython-only checks are labeled.")

Python 3.13.7 | implementation: CPython
All exercises use the standard library; CPython-only checks are labeled.


## Part I — Reference ownership and unexpected retention

The first exercises distinguish deletion of a *name* from destruction of an *object*. The event logs record what happened without depending on notebook output to keep target objects alive.

## Problem 01 — Two aliases, one object  
**Level:** Advanced foundation  

### Challenge
Write an experiment in which deleting one name does **not** call `__del__` because another name still refers to the object. Delete the last name, request a collection, and verify exactly one finalization event. Explain why `del` is not a call to `__del__`.

**Hint:** Keep an event list outside the class, and keep a `weakref.ref` rather than another strong reference.

### Solution 01 — reasoning
- `del first` unbinds a variable; it is not equivalent to `first.__del__()`.
- The alias still owns a strong reference, so the instance remains accessible.
- Once the instance becomes unreachable, CPython commonly invokes the finalizer immediately; the explicit collection also makes this simple example work on typical GC runtimes.
- Tests assert lifetime and events, never the raw memory address.

In [2]:
def alias_experiment():
    events = []

    class Tracked:
        def __del__(self):
            events.append("finalized")

    first = Tracked()
    probe = weakref.ref(first)
    second = first                # Another strong reference.
    del first                     # Delete a name, not the object.
    gc.collect()
    before = (probe() is not None, tuple(events))

    del second                    # Drop the remaining strong reference.
    gc.collect()
    after = (probe() is None, tuple(events))
    return before, after

alias_before, alias_after = alias_experiment()
print("After first deletion:", alias_before)
print("After final deletion:", alias_after)

After first deletion: (True, ())
After final deletion: (True, ('finalized',))


### Verification 01 — executable assertions

In [3]:
assert alias_before == (True, ())
assert alias_after == (True, ("finalized",))
print("PASS 01: one alias postpones finalization.")

PASS 01: one alias postpones finalization.


**Further challenge:** Add three aliases stored in different scopes and prove that all must be removed.

## Problem 02 — Reference-count deltas without unsafe pointers  
**Level:** Advanced foundation / CPython-specific  

### Challenge
Measure the *change* in reference count when adding/removing a second alias using `sys.getrefcount`. Why should you avoid asserting a hardcoded absolute count? Keep this exercise safe on non-CPython interpreters.

**Hint:** `sys.getrefcount(x)` temporarily adds its argument reference. Compare readings made the same way.

### Solution 02 — reasoning
- The observed absolute count includes implementation and evaluation details.
- A consistent extra alias contributes one additional strong reference on CPython.
- Comparing **deltas within one controlled function** is informative, but exact counts are not a cross-implementation API.
- Do not dereference a remembered address using `ctypes` after an object dies.

In [4]:
def refcount_delta_experiment():
    if not IS_CPYTHON:
        return None

    class Payload:
        pass

    item = Payload()
    initial = sys.getrefcount(item)
    alias = item
    after_alias = sys.getrefcount(item)
    del alias
    after_deletion = sys.getrefcount(item)
    return initial, after_alias, after_deletion

count_readings = refcount_delta_experiment()
print("CPython count readings:", count_readings)

CPython count readings: (2, 3, 2)


### Verification 02 — executable assertions

In [5]:
if IS_CPYTHON:
    baseline, added, removed = count_readings
    assert added - baseline == 1
    assert removed == baseline
else:
    assert count_readings is None
print("PASS 02: portable skip or CPython alias delta.")

PASS 02: portable skip or CPython alias delta.


**Further challenge:** Contrast the two safe quantities `sys.getrefcount(obj)` and `weakref.ref(obj)`; describe what each can and cannot tell you.

## Problem 03 — Invisible owners in containers and closures  
**Level:** Advanced  

### Challenge
Construct an instance whose finalizer cannot run after the local variable is deleted because (a) a list retains it and (b) a closure retains it. Prove that clearing only the list is insufficient.

**Hint:** A default argument or a closure cell can own a strong reference; delete each owner in sequence.

### Solution 03 — reasoning
- Each live container slot or closure cell can act as another owner.
- Clearing the list eliminates only one of the two hidden references.
- `weakref.ref` itself does not keep the referent alive; do not store the result `probe()` in a long-lived variable.

In [6]:
def hidden_owners_experiment():
    events = []

    class Target:
        def __del__(self):
            events.append("gone")

    target = Target()
    probe = weakref.ref(target)
    storage = [target]
    def make_reader(value):
        return lambda: value
    reader = make_reader(target)

    del target
    gc.collect()
    after_name = probe() is not None
    storage.clear()
    gc.collect()
    after_list = probe() is not None
    del reader
    gc.collect()
    after_closure = probe() is None
    return after_name, after_list, after_closure, tuple(events)

ownership_stages = hidden_owners_experiment()
print(ownership_stages)

(True, True, True, ('gone',))


### Verification 03 — executable assertions

In [7]:
assert ownership_stages == (True, True, True, ("gone",))
print("PASS 03: both hidden owners must disappear.")

PASS 03: both hidden owners must disappear.


**Further challenge:** Replace the closure with a `functools.partial` that retains the instance and repeat the ownership analysis.

## Problem 04 — A saved exception keeps its traceback frame alive  
**Level:** Advanced  

### Challenge
Raise an exception inside a function whose local variable refers to a tracked instance. Return the exception and a weak reference, but not the tracked instance. Show that the saved exception prevents collection; then release the exception and check that the instance can disappear.

**Hint:** An exception retains its `__traceback__`; a traceback retains frame objects and their local variables.

### Solution 04 — reasoning
- The chain is `exception → traceback → frame → frame locals → victim`.
- A traceback frame is especially significant in notebooks, debuggers, and exception caches.
- Do not claim that `del p` guarantees a finalizer: an unseen reference may still exist.

In [8]:
def make_error_and_probe(events):
    class Victim:
        def __del__(self):
            events.append("victim finalized")

    victim = Victim()
    probe = weakref.ref(victim)
    try:
        raise ValueError("retain my frame")
    except ValueError as caught:
        return probe, caught

def traceback_retention_experiment():
    events = []
    probe, saved_error = make_error_and_probe(events)
    gc.collect()
    held = probe() is not None
    frame_seen = saved_error.__traceback__ is not None
    del saved_error
    gc.collect()
    freed = probe() is None
    return held, frame_seen, freed, tuple(events)

traceback_result = traceback_retention_experiment()
print(traceback_result)

(True, True, True, ('victim finalized',))


### Verification 04 — executable assertions

In [9]:
assert traceback_result == (True, True, True, ("victim finalized",))
print("PASS 04: saved traceback retains a frame and its object.")

PASS 04: saved traceback retains a frame and its object.


**Further challenge:** Try `traceback.clear_frames(saved_error.__traceback__)` and investigate its limitations when the frame is still executing.

## Problem 05 — Inspect a traceback without retaining its frame  
**Level:** Expert diagnosis  

### Challenge
Walk a traceback chain and report only **safe summaries** (function names and whether a given local exists). Do not return traceback or frame objects, as that would recreate the retention problem being diagnosed.

**Hint:** Read `tb_frame.f_code.co_name` and `tb_frame.f_locals`, but return strings and booleans only.

### Solution 05 — reasoning
- Frames provide useful introspection but are themselves strong references when stored.
- A list of strings and booleans reports the diagnosis without owning the target.
- Avoid returning `frame.f_locals` directly: that dictionary may itself contain strong references to important objects.

In [10]:
def describe_traceback(error):
    summaries = []
    tb = error.__traceback__
    while tb is not None:
        frame = tb.tb_frame
        summaries.append({
            "function": frame.f_code.co_name,
            "has_victim": "victim" in frame.f_locals,
        })
        tb = tb.tb_next
    return summaries

def diagnose_traceback():
    events = []
    probe, saved_error = make_error_and_probe(events)
    summary = describe_traceback(saved_error)
    held = probe() is not None
    del saved_error
    gc.collect()
    return summary, held, probe() is None

tb_summary, tb_was_held, tb_now_free = diagnose_traceback()
print("Safe traceback summary:", tb_summary)

Safe traceback summary: [{'function': 'make_error_and_probe', 'has_victim': True}]


### Verification 05 — executable assertions

In [11]:
assert any(item["function"] == "make_error_and_probe" and item["has_victim"]
           for item in tb_summary)
assert tb_was_held and tb_now_free
print("PASS 05: diagnosis does not leak traceback frames.")

PASS 05: diagnosis does not leak traceback frames.


**Further challenge:** Extend the summary with source-line numbers using `tb.tb_lineno`; do not return the traceback nodes themselves.

## Problem 06 — Preserve an error message, not the exception object  
**Level:** Advanced  

### Challenge
Design a boundary that catches an exception, retains an informative formatted **string**, and returns a weak reference to an object that existed in the failing frame. Verify that storing only a string does not keep the frame or its object alive after the function returns.

**Hint:** Use `traceback.format_exception(...)`, join its strings, and do not return `exc` or `exc.__traceback__`.

### Solution 06 — reasoning
- The exception object is not returned or retained by an external cache.
- A formatted text representation can be held for reporting without retaining traceback frame locals.
- Beware: traceback formatting can expose sensitive details in real applications; redact before storing logs.

In [12]:
def capture_error_as_text(events):
    class Temp:
        def __del__(self):
            events.append("released")

    temp = Temp()
    probe = weakref.ref(temp)
    try:
        raise RuntimeError("example task failed")
    except RuntimeError as exc:
        message = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
    return probe, message

def text_boundary_experiment():
    events = []
    probe, message = capture_error_as_text(events)
    gc.collect()
    return ("RuntimeError: example task failed" in message,
            probe() is None, tuple(events), type(message).__name__)

text_boundary = text_boundary_experiment()
print(text_boundary)

(True, True, ('released',), 'str')


### Verification 06 — executable assertions

In [13]:
assert text_boundary == (True, True, ("released",), "str")
print("PASS 06: store an error summary without pinning the failed frame.")

PASS 06: store an error summary without pinning the failed frame.


**Further challenge:** Capture a structured record with exception class, message, and a timestamp, but no traceback or frame object.

## Part II — Cycles, resurrection, and finalizer errors

**Modern extension beyond the source lesson:** Python 3.4+ implements safe finalization of many cyclic isolates (PEP 442). A cycle containing `__del__` is **not automatically uncollectable**. Exact behavior can still depend on implementation, extension types, resurrection, and remaining references.

## Problem 07 — Collect a cycle whose nodes implement `__del__`  
**Level:** Expert  

### Challenge
Make two nodes point to each other and implement `__del__` on both. Drop outside references, explicitly call the cyclic collector, and check both weak references are dead. Do not assert the order in which finalizers run.

**Hint:** Build the cycle entirely inside a factory and return only weak references.

### Solution 07 — reasoning
- Pure reference counting cannot reclaim a cycle: each node is referenced by its peer.
- The cyclic garbage collector can collect this ordinary Python-level finalizable cycle on modern Python.
- The number of objects reported by `gc.collect()` is environment-dependent, so check weak references and event membership instead of hardcoding that count.

In [14]:
def create_finalizable_cycle(events):
    class Node:
        def __init__(self, label):
            self.label = label
            self.peer = None
        def __del__(self):
            events.append(self.label)

    left = Node("left")
    right = Node("right")
    left.peer, right.peer = right, left
    return weakref.ref(left), weakref.ref(right)

def cycle_experiment():
    events = []
    weak_left, weak_right = create_finalizable_cycle(events)
    gc.collect()
    return weak_left() is None, weak_right() is None, set(events), len(events)

cycle_result = cycle_experiment()
print(cycle_result)

(True, True, {'left', 'right'}, 2)


### Verification 07 — executable assertions

In [15]:
assert cycle_result[:2] == (True, True)
assert cycle_result[2:] == ({"left", "right"}, 2)
print("PASS 07: both finalizable cycle members were reclaimed.")

PASS 07: both finalizable cycle members were reclaimed.


**Further challenge:** Create a three-node ring and verify all three are reclaimed without relying on call order.

## Problem 08 — Turn automatic cyclic GC off, then restore it safely  
**Level:** Expert / CPython-specific  

### Challenge
Temporarily disable the **automatic** cyclic collector, build a cycle with no finalizers, and show that it stays alive on CPython until `gc.collect()` is called. Restore the original enabled state even if an assertion fails.

**Hint:** A `try/finally` protects interpreter-wide state; disabling automatic GC does not disable manual `gc.collect()`.

### Solution 08 — reasoning
- `gc.disable()` suspends scheduled cyclic collections, **not** the manual collector.
- On CPython, a strong-reference cycle survives reference-count decrements alone.
- Changing GC settings is process-wide; limit such experiments to controlled, single-threaded notebooks and always restore prior settings.

In [16]:
def make_plain_cycle():
    class Link:
        def __init__(self):
            self.next = None

    first, second = Link(), Link()
    first.next, second.next = second, first
    return weakref.ref(first), weakref.ref(second)

def paused_gc_experiment():
    was_enabled = gc.isenabled()
    gc.collect()                    # Reduce unrelated garbage first.
    try:
        gc.disable()
        first_ref, second_ref = make_plain_cycle()
        before_manual = (first_ref() is not None, second_ref() is not None)
        gc.collect()                # Explicit collection still works.
        after_manual = (first_ref() is None, second_ref() is None)
    finally:
        if was_enabled:
            gc.enable()
        else:
            gc.disable()
    return before_manual, after_manual, gc.isenabled() == was_enabled

paused_gc_result = paused_gc_experiment()
print(paused_gc_result)

((True, True), (True, True), True)


### Verification 08 — executable assertions

In [17]:
assert paused_gc_result[1] == (True, True)
assert paused_gc_result[2]
if IS_CPYTHON:
    assert paused_gc_result[0] == (True, True)
print("PASS 08: manual cycle collection and GC-state restoration.")

PASS 08: manual cycle collection and GC-state restoration.


**Further challenge:** Explain why this result does not imply that `gc.disable()` prevents every object from being freed.

## Problem 09 — Event-listener registry as a hidden strong owner  
**Level:** Advanced architecture  

### Challenge
A registry stores a bound method callback. Demonstrate that the registry keeps the callback's owner alive even after the main name is deleted. Implement `unsubscribe` and test that cleanup becomes possible.

**Hint:** A bound method retains its instance as `callback.__self__`.

### Solution 09 — reasoning
- The registry owns the **bound method**, and the bound method owns the instance.
- Unsubscribing alone is insufficient if another local variable still stores the same bound method.
- APIs should return clear subscription/unsubscription handles, or use carefully designed weak callbacks where appropriate.

In [18]:
class EventBus:
    def __init__(self):
        self._subscribers = []
    def subscribe(self, callback):
        self._subscribers.append(callback)
    def unsubscribe(self, callback):
        self._subscribers.remove(callback)
    def emit(self, event):
        for callback in tuple(self._subscribers):
            callback(event)

def event_bus_experiment():
    events = []
    bus = EventBus()

    class Listener:
        def notify(self, event):
            events.append(event)
        def __del__(self):
            events.append("finalized")

    listener = Listener()
    probe = weakref.ref(listener)
    bus.subscribe(listener.notify)
    bus.emit("hello")
    del listener
    gc.collect()
    kept_by_bus = probe() is not None
    callback = bus._subscribers[0]
    bus.unsubscribe(callback)
    del callback                 # Also release this local bound-method owner.
    gc.collect()
    return kept_by_bus, probe() is None, tuple(events)

bus_result = event_bus_experiment()
print(bus_result)

(True, True, ('hello', 'finalized'))


### Verification 09 — executable assertions

In [19]:
assert bus_result == (True, True, ("hello", "finalized"))
print("PASS 09: all bound-method owners must be dropped.")

PASS 09: all bound-method owners must be dropped.


**Further challenge:** Implement a registry using `weakref.WeakMethod` and prune dead callbacks safely.

## Problem 10 — Resurrection: a finalizer can save its own object  
**Level:** Expert  

### Challenge
Create a `__del__` method that stores `self` in an external list exactly once. Demonstrate that the object is still alive after its first finalization, then drop the rescue reference and show it can be deallocated without assuming a second `__del__` call.

**Hint:** The finalizer runs when the object is about to be destroyed; it may create a new strong reference.

### Solution 10 — reasoning
- Finalization and deallocation are separate steps; a finalizer may resurrect its object.
- On modern CPython, a finalizer is not normally called again after that resurrected object later dies. Avoid designing code that depends on repeated finalization.
- Resurrection makes life-cycle reasoning harder and is generally a poor resource-management mechanism.

In [20]:
def resurrection_experiment():
    shelf = []
    events = []

    class Phoenix:
        def __del__(self):
            events.append("finalized")
            if not shelf:
                shelf.append(self)      # A deliberate resurrection.

    def create():
        obj = Phoenix()
        return weakref.ref(obj)

    probe = create()
    gc.collect()
    rescued = (len(shelf) == 1 and probe() is not None)
    first_calls = len(events)
    shelf.clear()
    gc.collect()
    return rescued, first_calls, probe() is None, tuple(events)

resurrection_result = resurrection_experiment()
print(resurrection_result)

(True, 1, True, ('finalized',))


### Verification 10 — executable assertions

In [21]:
assert resurrection_result[0] is True
assert resurrection_result[1] == 1
assert resurrection_result[2] is True
assert resurrection_result[3] == ("finalized",)
print("PASS 10: resurrection delays deallocation; finalization was once.")

PASS 10: resurrection delays deallocation; finalization was once.


**Further challenge:** Discuss which invariants a resurrected object might violate if `__del__` partially tore down its state.

## Problem 11 — Catch an unraisable finalizer error correctly  
**Level:** Expert / controlled global hook  

### Challenge
Use `sys.unraisablehook` to capture **safe scalar details** when `__del__` raises an exception. Show that wrapping `del` in `try/except ValueError` does not catch that exception. Restore the original hook in `finally`.

**Hint:** Do not store `unraisable.exc_value` or `unraisable.object`; either could extend lifetimes.

### Solution 11 — reasoning
- An exception escaping `__del__` is **unraisable** to the deleting caller; Python reports it through its unraisable-exception machinery (normally stderr).
- Modern Python exposes `sys.unraisablehook` for controlled observation; older source material illustrates stderr redirection instead.
- Replacing a process-wide hook is for isolated demonstrations, not concurrent production code.

In [22]:
def unraisable_experiment():
    reports = []
    caller_caught = False
    original_hook = sys.unraisablehook

    def record(info):
        reports.append((info.exc_type.__name__, str(info.exc_value)))

    class Faulty:
        def __del__(self):
            raise ValueError("finalizer failed")

    try:
        sys.unraisablehook = record
        victim = Faulty()
        try:
            del victim
        except ValueError:
            caller_caught = True
        gc.collect()
    finally:
        sys.unraisablehook = original_hook
    return caller_caught, reports, sys.unraisablehook is original_hook

unraisable_result = unraisable_experiment()
print(unraisable_result)

(False, [('ValueError', 'finalizer failed')], True)


### Verification 11 — executable assertions

In [23]:
assert unraisable_result == (False, [("ValueError", "finalizer failed")], True)
print("PASS 11: finalizer exception was reported, not propagated.")

PASS 11: finalizer exception was reported, not propagated.


**Further challenge:** Implement a hook that delegates to the original hook after recording only scalar metadata; explain its logging trade-offs.

## Part III — Deterministic resource management

**Practical rule:** If correct behavior depends on *when* a resource is released or whether release failed, supply a named `close()` method and a context manager. Treat `__del__` as at most a best-effort fallback, not the transactional mechanism.

## Problem 12 — Idempotent close with explicit ownership  
**Level:** Advanced design  

### Challenge
Create a resource wrapper with `close()` that performs release **exactly once**, even if invoked multiple times. Demonstrate safe use under `try/finally`, including a failure in the work section. The resource must not require `__del__`.

**Hint:** Record the closed state before calling the underlying close operation, and use `finally` to guarantee the attempt.

### Solution 12 — reasoning
- `finally` provides a deterministic cleanup attempt on normal and exceptional control flow.
- An idempotent `close()` lets callers release safely without deciding whether some other path has already closed it.
- For fallible real-world close operations, define and document whether closing is retried or a failed close leaves an uncertain state.

In [24]:
class RecordingHandle:
    def __init__(self, events):
        self.events = events
        self.closed = False
    def write(self, data):
        if self.closed:
            raise ValueError("handle already closed")
        self.events.append(("write", data))
    def close(self):
        if not self.closed:
            self.closed = True
            self.events.append(("close", None))

def explicit_close_experiment():
    events = []
    resource = RecordingHandle(events)
    try:
        resource.write("payload")
        raise RuntimeError("work failed")
    except RuntimeError:
        events.append(("caught", None))
    finally:
        resource.close()
        resource.close()        # Demonstrate idempotence.
    return resource.closed, events

explicit_close_result = explicit_close_experiment()
print(explicit_close_result)

(True, [('write', 'payload'), ('caught', None), ('close', None)])


### Verification 12 — executable assertions

In [25]:
assert explicit_close_result == (
    True, [("write", "payload"), ("caught", None), ("close", None)]
)
print("PASS 12: explicit close attempted once despite repeated calls.")

PASS 12: explicit close attempted once despite repeated calls.


**Further challenge:** Make write fail for a second input; verify `finally` still invokes close once.

## Problem 13 — A class-based context manager propagates failures  
**Level:** Advanced design  

### Challenge
Implement a context manager that always closes an owned handle in `__exit__` and does **not** suppress exceptions from the `with` body. Verify release in both success and failure cases.

**Hint:** Return `False` from `__exit__`; never rely on `__del__` for timely closure.

### Solution 13 — reasoning
- `__enter__` acquires or exposes the resource; `__exit__` releases it for normal and exceptional exits.
- Returning `False` means exceptions from the body continue to propagate.
- Context managers are predictable about cleanup **attempts**; an actual `close()` can still fail and requires a documented error policy.

In [26]:
class ManagedHandle:
    def __init__(self, events):
        self._handle = RecordingHandle(events)
    def __enter__(self):
        return self._handle
    def __exit__(self, exc_type, exc_value, exc_tb):
        self._handle.close()
        return False

def class_cm_experiment():
    success_events = []
    with ManagedHandle(success_events) as handle:
        handle.write("ok")

    failure_events = []
    observed = None
    try:
        with ManagedHandle(failure_events) as handle:
            handle.write("before failure")
            raise LookupError("body error")
    except LookupError as exc:
        observed = str(exc)      # Store text, not the exception object.
    return success_events, failure_events, observed

cm_result = class_cm_experiment()
print(cm_result)

([('write', 'ok'), ('close', None)], [('write', 'before failure'), ('close', None)], 'body error')


### Verification 13 — executable assertions

In [27]:
assert cm_result[0] == [("write", "ok"), ("close", None)]
assert cm_result[1] == [("write", "before failure"), ("close", None)]
assert cm_result[2] == "body error"
print("PASS 13: success and failure both close; body error propagates.")

PASS 13: success and failure both close; body error propagates.


**Further challenge:** Add a `closed` read-only property and prove the handle cannot be written after leaving `with`.

## Problem 14 — Generator-based context manager and exception propagation  
**Level:** Advanced design  

### Challenge
Implement the same lifecycle using `@contextlib.contextmanager`. Demonstrate that code after `yield` runs in a `finally` suite even if the body raises. Do not suppress the original exception.

**Hint:** A generator context manager usually uses `try: yield resource` followed by `finally: resource.close()`.

### Solution 14 — reasoning
- The `yield` divides setup and teardown; `finally` is the cleanup guarantee.
- It is easy to accidentally suppress an error when catching exceptions inside a context-manager generator. Either re-raise, or use only a `finally` for unconditional cleanup.

In [28]:
@contextlib.contextmanager
def acquire_recording_handle(events):
    handle = RecordingHandle(events)
    try:
        yield handle
    finally:
        handle.close()

def generator_cm_experiment():
    events = []
    message = None
    try:
        with acquire_recording_handle(events) as handle:
            handle.write("processed")
            raise ArithmeticError("computation failed")
    except ArithmeticError as exc:
        message = str(exc)
    return events, message

generator_cm_result = generator_cm_experiment()
print(generator_cm_result)

([('write', 'processed'), ('close', None)], 'computation failed')


### Verification 14 — executable assertions

In [29]:
assert generator_cm_result == (
    [("write", "processed"), ("close", None)], "computation failed"
)
print("PASS 14: generator manager releases resource without swallowing the error.")

PASS 14: generator manager releases resource without swallowing the error.


**Further challenge:** Add a validation error before `yield`; explain why `__exit__` cannot clean up a resource whose `__enter__` never succeeded unless acquisition handles its own rollback.

## Problem 15 — ExitStack and partial acquisition rollback  
**Level:** Expert design  

### Challenge
Build three resources that can fail upon entering. Use `contextlib.ExitStack` to acquire them in sequence. When the third resource fails to enter, prove that earlier successful acquisitions close in reverse order. Do not assert that the failing resource closes unless it arranged its own rollback.

**Hint:** Call `stack.enter_context(...)` in a `with ExitStack() as stack` block.

### Solution 15 — reasoning
- `ExitStack` registers exits only after a successful entry.
- The acquired resources are unwound last-in, first-out (`B`, then `A`) when `C` fails.
- If `C` allocated something *before* rejecting entry, `C.__enter__` must clean up its partially acquired state itself.

In [30]:
class NamedResource:
    def __init__(self, name, events, fail_enter=False):
        self.name = name
        self.events = events
        self.fail_enter = fail_enter
    def __enter__(self):
        self.events.append("open:" + self.name)
        if self.fail_enter:
            self.events.append("reject:" + self.name)
            raise OSError("cannot acquire " + self.name)
        return self
    def __exit__(self, exc_type, exc_value, exc_tb):
        self.events.append("close:" + self.name)
        return False

def rollback_experiment():
    events = []
    message = None
    try:
        with contextlib.ExitStack() as stack:
            stack.enter_context(NamedResource("A", events))
            stack.enter_context(NamedResource("B", events))
            stack.enter_context(NamedResource("C", events, fail_enter=True))
            events.append("unreachable")
    except OSError as exc:
        message = str(exc)
    return events, message

rollback_result = rollback_experiment()
print(rollback_result)

(['open:A', 'open:B', 'open:C', 'reject:C', 'close:B', 'close:A'], 'cannot acquire C')


### Verification 15 — executable assertions

In [31]:
assert rollback_result == (
    ["open:A", "open:B", "open:C", "reject:C", "close:B", "close:A"],
    "cannot acquire C",
)
print("PASS 15: partial acquisition rolls back in reverse order.")

PASS 15: partial acquisition rolls back in reverse order.


**Further challenge:** Use `ExitStack.callback` for an old-style API that has `open()`/`close()` but no context-manager support.

## Part IV — Weak references and finalization traps

A weak reference observes lifetime without extending it. `weakref.finalize` offers a callback-based fallback, but the registered callback must **not strongly capture the watched object**. None of these mechanisms replaces deterministic explicit close.

## Problem 16 — Use a weak reference as an observation probe  
**Level:** Advanced foundation  

### Challenge
Track a live instance and observe its later death with a weak reference. Also register a weakref callback that records an event without trying to access the dead referent. Do not store a bound callback that keeps the target alive.

**Hint:** Pass a standalone callback closing over an event list, not the referent.

### Solution 16 — reasoning
- `weakref.ref(obj)` does not create an owning reference.
- At callback time, the object has already become unreachable to the weak reference; `dead_ref()` returns `None`.
- Weak callbacks are notification mechanisms, not replacements for a guaranteed cleanup protocol.

In [32]:
def weak_probe_experiment():
    events = []

    class Target:
        pass

    def callback(dead_ref):
        events.append(("callback", dead_ref() is None))

    obj = Target()
    probe = weakref.ref(obj, callback)
    before = probe() is obj
    del obj
    gc.collect()
    after = probe() is None
    return before, after, events

weak_probe_result = weak_probe_experiment()
print(weak_probe_result)

(True, True, [('callback', True)])


### Verification 16 — executable assertions

In [33]:
assert weak_probe_result == (True, True, [("callback", True)])
print("PASS 16: weak reference observes rather than owns.")

PASS 16: weak reference observes rather than owns.


**Further challenge:** Design a cache backed by `weakref.WeakValueDictionary` and explain when entries can disappear.

## Problem 17 — Design a safe weakref.finalize fallback  
**Level:** Expert  

### Challenge
Arrange for a separate handle to close when its owner disappears. Use `weakref.finalize(owner, handle.close)` safely: the handle must not reference the owner. Verify that the fallback runs exactly once and that its `alive` property becomes false.

**Hint:** The callback may strongly hold a separate handle; that is fine if the handle does **not** point back to the watched owner.

### Solution 17 — reasoning
- `weakref.finalize` internally retains the callback and arguments until invocation or detachment.
- Registering a callback on an **independent handle** is safe because that callback has no strong path back to `owner`.
- Fallback callbacks can run too late for correctness. A context manager should still perform normal cleanup.

In [34]:
class ExternalHandle:
    def __init__(self, events):
        self.events = events
        self.closed = False
    def close(self):
        if not self.closed:
            self.closed = True
            self.events.append("closed external handle")

class Owner:
    pass

def finalize_safe_experiment():
    events = []
    handle = ExternalHandle(events)
    owner = Owner()
    probe = weakref.ref(owner)
    fallback = weakref.finalize(owner, handle.close)
    initially_alive = fallback.alive
    del owner
    gc.collect()
    return (initially_alive, probe() is None, handle.closed,
            fallback.alive, tuple(events))

finalize_safe_result = finalize_safe_experiment()
print(finalize_safe_result)

(True, True, True, False, ('closed external handle',))


### Verification 17 — executable assertions

In [35]:
assert finalize_safe_result == (
    True, True, True, False, ("closed external handle",)
)
print("PASS 17: safe callback cannot retain the watched owner.")

PASS 17: safe callback cannot retain the watched owner.


**Further challenge:** Call `fallback()` explicitly before deleting the owner; test that it executes once and does not run a second time.

## Problem 18 — Spot and repair the finalize-bound-method trap  
**Level:** Expert debugging  

### Challenge
Demonstrate that `weakref.finalize(obj, obj.cleanup)` retains `obj` through its bound method, preventing it from becoming unreachable. Repair the design with `detach()` and show how to avoid the trap in new code.

**Hint:** A bound method has a strong `__self__` reference; `finalize` retains the callback. `detach()` cancels the registration.

### Solution 18 — reasoning
- The strong-reference chain is `finalize registry → bound cleanup method → obj`.
- This is a genuine retention trap, not a dependable finalization pattern.
- `detach()` removes the finalizer registration; use a callback operating on an **independent resource** (as in Problem 17) for new code.

In [36]:
class LeakyOwner:
    def cleanup(self):
        pass

def finalize_bound_method_trap():
    obj = LeakyOwner()
    probe = weakref.ref(obj)
    registration = weakref.finalize(obj, obj.cleanup)   # ANTI-PATTERN
    del obj
    gc.collect()
    pinned = probe() is not None and registration.alive

    # Cancel the callback and discard the tuple detach() returns immediately.
    registration.detach()
    gc.collect()
    released = probe() is None and not registration.alive
    return pinned, released

trap_result = finalize_bound_method_trap()
print(trap_result)

(True, True)


### Verification 18 — executable assertions

In [37]:
assert trap_result == (True, True)
print("PASS 18: bound method traps referent; detaching breaks chain.")

PASS 18: bound method traps referent; detaching breaks chain.


**Further challenge:** Try an apparently harmless `lambda: obj.cleanup()` closure and explain why it is likewise unsafe if it captures the owner strongly.

## Problem 19 — Weak-value cache with deterministic checks  
**Level:** Expert architecture  

### Challenge
Build a small cache in which values are owned by the caller, not by the cache. Show that a cache lookup returns the instance while a strong owner exists and that the cache entry disappears once the owner is gone. Avoid asserting a precise collection time without `gc.collect()`.

**Hint:** Use `weakref.WeakValueDictionary`; cache keys remain strong, cached values are weak.

### Solution 19 — reasoning
- Weak-value caches are appropriate when cache membership should not extend an object's lifetime.
- A cache miss after collection is expected: the caller must be able to recompute or reload the value.
- Do not expect weak-value caches to store immutable built-in types, such as plain `int` or `str`, which generally do not support weak references.

In [38]:
class CachedItem:
    def __init__(self, name):
        self.name = name

def weak_cache_experiment():
    cache = weakref.WeakValueDictionary()
    item = CachedItem("report")
    cache["report"] = item
    probe = weakref.ref(item)
    present = (cache.get("report") is item)
    del item
    gc.collect()
    absent = (probe() is None and "report" not in cache)
    return present, absent

cache_result = weak_cache_experiment()
print(cache_result)

(True, True)


### Verification 19 — executable assertions

In [39]:
assert cache_result == (True, True)
print("PASS 19: cache is an observer, not an owner.")

PASS 19: cache is an observer, not an owner.


**Further challenge:** Explain why `cache.get(key)` may briefly create a new strong reference while you use its return value.

## Part V — Concurrency and integrated lifecycle design

The final two problems combine ownership, error propagation, and predictable cleanup. Thread scheduling and the thread that executes `__del__` are not reliable resource-management tools.

## Problem 20 — Make close idempotent across many threads  
**Level:** Expert concurrency  

### Challenge
Several threads may call `close()` on the same resource. Implement a lock-protected, once-only close transition, collect failures from workers, and verify that the underlying release action was performed once. Do **not** use finalization timing for synchronization.

**Hint:** Protect the state check and close transition with one `threading.Lock`.

### Solution 20 — reasoning
- A mutex protects the check-and-set operation; only one caller performs the release transition.
- Release under the lock is simple but may block other callers if the actual I/O operation is slow. More elaborate lifecycle state machines can distinguish `OPEN`, `CLOSING`, and `CLOSED` and coordinate waiters.
- There is no guarantee that `__del__` runs on a preferred thread, so finalizers should not orchestrate concurrent release.

In [40]:
class ThreadSafeResource:
    def __init__(self):
        self._lock = threading.Lock()
        self._closed = False
        self.release_calls = 0
    def close(self):
        with self._lock:
            if self._closed:
                return False
            self._closed = True
            self.release_calls += 1   # Represents underlying release under the lock.
            return True
    @property
    def closed(self):
        with self._lock:
            return self._closed

def concurrent_close_experiment(workers=12):
    resource = ThreadSafeResource()
    outcomes = []
    outcome_lock = threading.Lock()
    def worker():
        result = resource.close()
        with outcome_lock:
            outcomes.append(result)
    threads = [threading.Thread(target=worker) for _ in range(workers)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    return resource.closed, resource.release_calls, outcomes

concurrent_result = concurrent_close_experiment()
print("closed, release_calls, winning callers:",
      concurrent_result[0], concurrent_result[1], sum(concurrent_result[2]))

closed, release_calls, winning callers: True 1 1


### Verification 20 — executable assertions

In [41]:
closed, release_calls, outcomes = concurrent_result
assert closed is True
assert release_calls == 1
assert len(outcomes) == 12
assert outcomes.count(True) == 1
assert outcomes.count(False) == 11
print("PASS 20: exactly one thread owns the close transition.")

PASS 20: exactly one thread owns the close transition.


**Further challenge:** Implement `OPEN → CLOSING → CLOSED` with a `Condition` so other callers wait until a slow close operation completes.

## Problem 21 — Capstone: transaction-like acquisition and release  
**Level:** Expert capstone  

### Challenge
Write a `Pipeline` using `ExitStack`: acquire three named stages, perform work, and always release them in reverse order. Support failure (a) when acquiring a stage and (b) during processing. Return only event logs and sanitized error strings. Verify all three scenarios: success, acquisition failure, and processing failure.

**Hint:** Factor the execution into a small function and let `ExitStack` manage only successfully entered resources.

### Solution 21 — reasoning
- All acquired contexts are released in reverse order, even if processing raises.
- A stage that fails during `__enter__` never registers an `__exit__`; it must roll back its own partial setup.
- The error report is plain text rather than a cached exception with a traceback.
- `work:commit` is an illustrative event, **not** a real database transaction: actual database commit/rollback semantics must be implemented explicitly.

In [42]:
class Pipeline:
    def __init__(self, names):
        self.names = tuple(names)
    def run(self, *, reject=None, fail_work=False):
        events = []
        error_text = None
        try:
            with contextlib.ExitStack() as stack:
                for name in self.names:
                    stack.enter_context(NamedResource(
                        name, events, fail_enter=(name == reject)
                    ))
                events.append("work:start")
                if fail_work:
                    raise RuntimeError("processing stopped")
                events.append("work:commit")
        except (OSError, RuntimeError) as exc:
            error_text = f"{type(exc).__name__}: {exc}"
        return events, error_text

pipeline = Pipeline(("load", "transform", "store"))
pipeline_success = pipeline.run()
pipeline_enter_failure = pipeline.run(reject="transform")
pipeline_work_failure = pipeline.run(fail_work=True)
for name, result in (("success", pipeline_success),
                     ("acquisition error", pipeline_enter_failure),
                     ("processing error", pipeline_work_failure)):
    print(name, "=>", result)

success => (['open:load', 'open:transform', 'open:store', 'work:start', 'work:commit', 'close:store', 'close:transform', 'close:load'], None)
acquisition error => (['open:load', 'open:transform', 'reject:transform', 'close:load'], 'OSError: cannot acquire transform')
processing error => (['open:load', 'open:transform', 'open:store', 'work:start', 'close:store', 'close:transform', 'close:load'], 'RuntimeError: processing stopped')


### Verification 21 — executable assertions

In [43]:
assert pipeline_success == (
    ["open:load", "open:transform", "open:store", "work:start",
     "work:commit", "close:store", "close:transform", "close:load"], None
)
assert pipeline_enter_failure == (
    ["open:load", "open:transform", "reject:transform", "close:load"],
    "OSError: cannot acquire transform"
)
assert pipeline_work_failure == (
    ["open:load", "open:transform", "open:store", "work:start",
     "close:store", "close:transform", "close:load"],
    "RuntimeError: processing stopped"
)
print("PASS 21: success, partial acquisition, and body failure all handled.")

PASS 21: success, partial acquisition, and body failure all handled.


**Further challenge:** Extend the pipeline with a separate reversible `commit()`/`rollback()` interface; write a test in which rollback itself raises and document which error your API preserves.

## Quick-reference decision guide

| Need | Prefer | Avoid |
|---|---|---|
| Guarantee cleanup attempt at a known boundary | `with` / `try...finally` / explicit `close()` | Depending on `__del__` timing |
| Acquire several independent resources | `contextlib.ExitStack` | Manual cleanup with missing exception paths |
| Observe whether an object survives | `weakref.ref` | Dereferencing stale `id()` values with `ctypes` |
| Hold a cache without owning values | `WeakValueDictionary` | Accidental strong-reference caches |
| Fallback when an owner vanishes | `weakref.finalize(owner, independent_handle.close)` | `weakref.finalize(owner, owner.close)` |
| Diagnose mysterious retention | Inspect reference owners and traceback chains | Assuming `del variable` destroys an instance |
| Report exceptions from a finalizer | Understand unraisable reporting; avoid raising | Expecting caller `try/except` to catch them |
| Handle simultaneous close calls | Explicit locked lifecycle | Coordinating via `__del__` |

### Review questions (explain without running code)
1. Why is the sentence “`del x` calls the destructor” technically wrong?
2. What exact ownership path does a stored exception create through its traceback?
3. Why does `gc.disable()` not block an explicit call to `gc.collect()`?
4. What changed in Python 3.4+ concerning objects with `__del__` inside ordinary cycles?
5. What are the two ownership bugs in `weakref.finalize(owner, owner.close)` and `registry.append(owner.method)`?
6. What is the difference between a guaranteed *cleanup attempt* and a guaranteed successful release?

**Answers:** (1) `del` deletes a binding; other owners may remain. (2) Exception → traceback → frame → local object. (3) The setting disables automatic cyclic runs only. (4) PEP 442 permits safe finalization and collection of many cycles with finalizers. (5) Both retain bound methods, which strongly retain the owner. (6) Cleanup itself can raise or fail even when the language ensures the teardown code executes.

### Reference pointers
- **Source lesson:** supplied `__del__` tutorial (aliasing, traceback retention, ignored finalizer errors, and context managers).
- **Additional language and standard-library background (not copied from the source lesson):** Python documentation for [Data model: `object.__del__`](https://docs.python.org/3/reference/datamodel.html#object.__del__), [Garbage collector](https://docs.python.org/3/library/gc.html), [Weak references](https://docs.python.org/3/library/weakref.html), [Context manager utilities](https://docs.python.org/3/library/contextlib.html), and [PEP 442](https://peps.python.org/pep-0442/).

*Notebook-end checkpoint:* All 21 verification cells should print `PASS` when run in a normal CPython kernel. No exercise depends on a particular memory address, destructor ordering, or finalization during interpreter shutdown.